In [ ]:
import os
import json
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, average_precision_score, f1_score, precision_recall_curve

# ====== CONFIG ======
IN_FILE = "risultati_kart-trans-token_features.csv"  # <-- cambia se diverso
MODEL_OUT = "catboost_td_distilled.cbm"
META_OUT = "catboost_td_distilled_meta.json"
TEXT_COL = "orig_text"
TARGET_COL = "logit_td"  # distillazione fedele
LABEL_COL = "label"      # opzionale per metriche classificazione
RANDOM_STATE = 42
TEST_SIZE = 0.2

# ====== LOAD ======
df = pd.read_csv(IN_FILE)
df.columns = [c.strip() for c in df.columns]

# sanity checks
required = [TEXT_COL, TARGET_COL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Mancano colonne richieste: {missing}. Colonne disponibili: {df.columns.tolist()}")

# pulizia base
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("").str.strip()
df = df[df[TEXT_COL].str.len() > 0].copy()

# target
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL]).copy()

# ====== FEATURE COLUMNS ======
# numeriche: tutto ciò che inizia con sal_ + contatori token
num_cols = [c for c in df.columns if c.startswith("sal_") or c.startswith("n_tokens_")]
# rimuovi target/label se per caso matchano
num_cols = [c for c in num_cols if c not in [TARGET_COL, LABEL_COL]]
# se alcune numeriche hanno NaN, CatBoost li gestisce, ma meglio riempire per stabilità
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
# per i NaN: metti 0 sulle feature di conteggio/somme, e mediana sulle altre
for c in num_cols:
    if (
        c.startswith("n_tokens_")
        or c.endswith("_sum")
        or c.endswith("_cnt")
        or c.endswith("_pct")
        or c.endswith("_share")
    ):
        df[c] = df[c].fillna(0.0)
    else:
        df[c] = df[c].fillna(df[c].median())

feature_cols = [TEXT_COL]

# ====== SPLIT ======
train_df, valid_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[LABEL_COL] if LABEL_COL in df.columns else None
)

train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df[TARGET_COL],
    text_features=[TEXT_COL]
)
valid_pool = Pool(
    data=valid_df[feature_cols],
    label=valid_df[TARGET_COL],
    text_features=[TEXT_COL]
)

# ====== TRAIN ======
model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=5000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    early_stopping_rounds=200,
    verbose=200
)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

# ====== EVAL (distillation fidelity) ======
pred_logit = model.predict(valid_pool)
rmse = np.sqrt(mean_squared_error(valid_df[TARGET_COL].values, pred_logit))
print(f"\nRMSE(logit) valid: {rmse:.6f}")

# ====== EVAL (optional classification vs label) ======
# sigmoid
p_hat = 1.0 / (1.0 + np.exp(-pred_logit))
if LABEL_COL in valid_df.columns and valid_df[LABEL_COL].notna().any():
    y_true = valid_df[LABEL_COL].astype(int).values
    # PR-AUC
    pr_auc = average_precision_score(y_true, p_hat)
    print(f"PR-AUC valid (vs label): {pr_auc:.6f}")
    # soglia ottima per F1 su validation
    precisions, recalls, thresholds = precision_recall_curve(y_true, p_hat)
    f1s = (2 * precisions * recalls) / (precisions + recalls + 1e-12)
    best_idx = int(np.nanargmax(f1s))
    best_thr = float(thresholds[max(best_idx - 1, 0)]) if len(thresholds) else 0.5
    y_pred = (p_hat >= best_thr).astype(int)
    f1 = f1_score(y_true, y_pred)
    print(f"Best threshold (F1): {best_thr:.6f} | F1 valid: {f1:.6f}")
else:
    pr_auc = None
    best_thr = None
    f1 = None
    print("Label non presente (o vuota) -> salto metriche classificazione.")

# ====== SAVE ======
model.save_model(MODEL_OUT)
meta = {
    "input_file": IN_FILE,
    "text_col": TEXT_COL,
    "target_col": TARGET_COL,
    "label_col": LABEL_COL if LABEL_COL in df.columns else None,
    "num_cols": num_cols,
    "feature_cols": feature_cols,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "rmse_valid_logit": rmse,
    "pr_auc_valid": pr_auc,
    "best_threshold_valid": best_thr,
    "f1_valid": f1,
    "model_file": MODEL_OUT
}
with open(META_OUT, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("\nSalvato modello:", MODEL_OUT)
print("Salvati metadata:", META_OUT)
print("Num features:", len(feature_cols), "| Numeric:", len(num_cols))


In [ ]:
import os
import json
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, average_precision_score, f1_score, precision_recall_curve

# ====== CONFIG ======
IN_FILE = "risultati_kart-trans-token_features.csv"  # <-- cambia se diverso
TEXT_COL = "orig_text"
TARGET_COL = "logit_td"  # distillazione fedele
LABEL_COL = "label"      # opzionale per metriche classificazione
RANDOM_STATE = 42
TEST_SIZE = 0.2

# ====== LOAD ======
df = pd.read_csv(IN_FILE)
df.columns = [c.strip() for c in df.columns]

# sanity checks
required = [TEXT_COL, TARGET_COL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Mancano colonne richieste: {missing}. Colonne disponibili: {df.columns.tolist()}")

# pulizia base
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("").str.strip()
df = df[df[TEXT_COL].str.len() > 0].copy()

# target
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL]).copy()

# ====== FEATURE COLUMNS ======
# numeriche: tutto ciò che inizia con sal_ + contatori token
num_cols = [c for c in df.columns if c.startswith("sal_") or c.startswith("n_tokens_")]
# rimuovi target/label se per caso matchano
num_cols = [c for c in num_cols if c not in [TARGET_COL, LABEL_COL]]
# se alcune numeriche hanno NaN, CatBoost li gestisce, ma meglio riempire per stabilità
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
# per i NaN: metti 0 sulle feature di conteggio/somme, e mediana sulle altre
for c in num_cols:
    if (
        c.startswith("n_tokens_")
        or c.endswith("_sum")
        or c.endswith("_cnt")
        or c.endswith("_pct")
        or c.endswith("_share")
    ):
        df[c] = df[c].fillna(0.0)
    else:
        df[c] = df[c].fillna(df[c].median())

feature_cols = [TEXT_COL] + num_cols

# ====== SPLIT ======
train_df, valid_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[LABEL_COL] if LABEL_COL in df.columns else None
)

train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df[TARGET_COL],
    text_features=[TEXT_COL]
)
valid_pool = Pool(
    data=valid_df[feature_cols],
    label=valid_df[TARGET_COL],
    text_features=[TEXT_COL]
)

# ====== TRAIN ======
model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=5000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    early_stopping_rounds=200,
    verbose=200
)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

# ====== EVAL (distillation fidelity) ======
pred_logit = model.predict(valid_pool)
rmse = np.sqrt(mean_squared_error(valid_df[TARGET_COL].values, pred_logit))
print(f"\nRMSE(logit) valid: {rmse:.6f}")

# ====== EVAL (optional classification vs label) ======
# sigmoid
p_hat = 1.0 / (1.0 + np.exp(-pred_logit))
if LABEL_COL in valid_df.columns and valid_df[LABEL_COL].notna().any():
    y_true = valid_df[LABEL_COL].astype(int).values
    # PR-AUC
    pr_auc = average_precision_score(y_true, p_hat)
    print(f"PR-AUC valid (vs label): {pr_auc:.6f}")
    # soglia ottima per F1 su validation
    precisions, recalls, thresholds = precision_recall_curve(y_true, p_hat)
    f1s = (2 * precisions * recalls) / (precisions + recalls + 1e-12)
    best_idx = int(np.nanargmax(f1s))
    best_thr = float(thresholds[max(best_idx - 1, 0)]) if len(thresholds) else 0.5
    y_pred = (p_hat >= best_thr).astype(int)
    f1 = f1_score(y_true, y_pred)
    print(f"Best threshold (F1): {best_thr:.6f} | F1 valid: {f1:.6f}")
else:
    pr_auc = None
    best_thr = None
    f1 = None
    print("Label non presente (o vuota) -> salto metriche classificazione.")


In [ ]:
import os
import json
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, average_precision_score, f1_score, precision_recall_curve

# ====== CONFIG ======
IN_FILE = "merged_id_text_label.csv"  # <-- cambia se diverso
MODEL_OUT = "catboost_td_distilled-v2.cbm"
META_OUT = "catboost_td_distilled_meta-v2.json"
TEXT_COL = "orig_text"
TARGET_COL = "logit_td"  # distillazione fedele
LABEL_COL = "label"      # opzionale per metriche classificazione
RANDOM_STATE = 42
TEST_SIZE = 0.2

# ====== LOAD ======
df = pd.read_csv(IN_FILE)
df.columns = [c.strip() for c in df.columns]

# sanity checks
required = [TEXT_COL, TARGET_COL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Mancano colonne richieste: {missing}. Colonne disponibili: {df.columns.tolist()}")

# pulizia base
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("").str.strip()
df = df[df[TEXT_COL].str.len() > 0].copy()

# target
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL]).copy()

# ====== FEATURE COLUMNS ======
num_cols = []
feature_cols = [TEXT_COL]

# ====== SPLIT ======
train_df, valid_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[LABEL_COL] if LABEL_COL in df.columns else None
)

train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df[TARGET_COL],
    text_features=[TEXT_COL]
)
valid_pool = Pool(
    data=valid_df[feature_cols],
    label=valid_df[TARGET_COL],
    text_features=[TEXT_COL]
)

# ====== TRAIN ======
model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=5000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    early_stopping_rounds=200,
    verbose=200
)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

# ====== EVAL (distillation fidelity) ======
pred_logit = model.predict(valid_pool)
rmse = np.sqrt(mean_squared_error(valid_df[TARGET_COL].values, pred_logit))
print(f"\nRMSE(logit) valid: {rmse:.6f}")

# ====== EVAL (optional classification vs label) ======
# sigmoid
p_hat = 1.0 / (1.0 + np.exp(-pred_logit))
if LABEL_COL in valid_df.columns and valid_df[LABEL_COL].notna().any():
    y_true = valid_df[LABEL_COL].astype(int).values
    # PR-AUC
    pr_auc = average_precision_score(y_true, p_hat)
    print(f"PR-AUC valid (vs label): {pr_auc:.6f}")
    # soglia ottima per F1 su validation
    precisions, recalls, thresholds = precision_recall_curve(y_true, p_hat)
    f1s = (2 * precisions * recalls) / (precisions + recalls + 1e-12)
    best_idx = int(np.nanargmax(f1s))
    best_thr = float(thresholds[max(best_idx - 1, 0)]) if len(thresholds) else 0.5
    y_pred = (p_hat >= best_thr).astype(int)
    f1 = f1_score(y_true, y_pred)
    print(f"Best threshold (F1): {best_thr:.6f} | F1 valid: {f1:.6f}")
else:
    pr_auc = None
    best_thr = None
    f1 = None
    print("Label non presente (o vuota) -> salto metriche classificazione.")

# ====== SAVE ======
model.save_model(MODEL_OUT)
meta = {
    "input_file": IN_FILE,
    "text_col": TEXT_COL,
    "target_col": TARGET_COL,
    "label_col": LABEL_COL if LABEL_COL in df.columns else None,
    "num_cols": num_cols,
    "feature_cols": feature_cols,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "rmse_valid_logit": rmse,
    "pr_auc_valid": pr_auc,
    "best_threshold_valid": best_thr,
    "f1_valid": f1,
    "model_file": MODEL_OUT
}
with open(META_OUT, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("\nSalvato modello:", MODEL_OUT)
print("Salvati metadata:", META_OUT)
print("Num features:", len(feature_cols), "| Numeric:", len(num_cols))


In [ ]:
import json
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, average_precision_score, f1_score, precision_recall_curve
import re

# ====== CONFIG ======
IN_FILE = "merged_id_text_label.csv"
MODEL_OUT = "catboost_td_distilled-v3-cheap.cbm"
META_OUT = "catboost_td_distilled_meta-v3-cheap.json"
TEXT_COL = "orig_text"
TARGET_COL = "logit_td"  # distillazione fedele (teacher)
LABEL_COL = "label"      # opzionale per metriche classificazione
RANDOM_STATE = 42
TEST_SIZE = 0.2
# oversampling “soft” dei borderline (vicini a 0)
USE_BORDERLINE_WEIGHTS = True
W_MAX = 5.0  # peso massimo vicino a logit=0 (es. 3..10)

# ====== LOAD ======
df = pd.read_csv(IN_FILE)
df.columns = [c.strip() for c in df.columns]
required = [TEXT_COL, TARGET_COL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Mancano colonne richieste: {missing}. Colonne disponibili: {df.columns.tolist()}")
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("").str.strip()
df = df[df[TEXT_COL].str.len() > 0].copy()
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL]).copy()

# ====== CHEAP FEATURES ======
KW = ["todo", "fixme", "hack", "workaround", "temp", "quick"]
_re_word = re.compile(r"\b\w+\b", re.UNICODE)
_re_upper = re.compile(r"[A-Z]")
_re_digit = re.compile(r"\d")
_re_punct = re.compile(r"[^\w\s]")
_re_snake = re.compile(r"\b[a-z]+_[a-z0-9_]+\b")
_re_camel = re.compile(r"\b[a-z]+[A-Z][A-Za-z0-9]*\b")

def kw_count_word(tl: str, kw: str) -> int:
    # conta a parola intera (evita match dentro altre parole)
    return len(re.findall(rf"\b{re.escape(kw)}\b", tl))

def cheap_features(text: str) -> dict:
    t = "" if text is None else str(text)
    tl = t.lower()
    n_chars = len(t)
    words = _re_word.findall(t)
    n_words = len(words) if words else 0
    n_upper = len(_re_upper.findall(t))
    n_digit = len(_re_digit.findall(t))
    n_punct = len(_re_punct.findall(t))
    pct_upper = n_upper / n_chars if n_chars else 0.0
    pct_digit = n_digit / n_chars if n_chars else 0.0
    pct_punct = n_punct / n_chars if n_chars else 0.0
    feats = {
        "n_chars": n_chars,
        "n_words": n_words,
        "pct_upper": pct_upper,
        "pct_digit": pct_digit,
        "pct_punct": pct_punct,
        "has_should": int(" should " in f" {tl} "),
        "has_later": int(" later " in f" {tl} "),
        "has_for_now": int(" for now " in tl),
        "has_snake_case": int(_re_snake.search(t) is not None),
        "has_camel_case": int(_re_camel.search(t) is not None),
        "has_path": int(("/" in t) or ("\\" in t)),
        "has_equals": int("=" in t),
        "has_brackets": int(any(ch in t for ch in "(){}[]")),
    }
    for k in KW:
        cnt = kw_count_word(tl, k)
        feats[f"kw_{k}_cnt"] = cnt
        feats[f"kw_{k}_has"] = int(cnt > 0)
    return feats

df_feats = df.copy()
cheap = df_feats[TEXT_COL].apply(cheap_features).apply(pd.Series)
df_feats = pd.concat([df_feats, cheap], axis=1)
num_cols = [c for c in df_feats.columns if c.startswith(("n_", "pct_", "kw_", "has_"))]
feature_cols = [TEXT_COL] + num_cols

# ====== SPLIT ======
strat = df_feats[LABEL_COL] if (LABEL_COL in df_feats.columns and df_feats[LABEL_COL].notna().any()) else None
train_df, valid_df = train_test_split(
    df_feats,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=strat
)

# ====== WEIGHTS (borderline focus) ======
train_w = None
if USE_BORDERLINE_WEIGHTS:
    a = np.abs(train_df[TARGET_COL].values)
    # peso continuo: vicino a 0 -> ~W_MAX, lontano -> ~1
    train_w = 1.0 + (W_MAX - 1.0) * np.exp(-a)

# ====== POOLS ======
train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df[TARGET_COL],
    text_features=[TEXT_COL],
    weight=train_w
)
valid_pool = Pool(
    data=valid_df[feature_cols],
    label=valid_df[TARGET_COL],
    text_features=[TEXT_COL]
)

# ====== TRAIN ======
model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=6000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    early_stopping_rounds=200,
    verbose=200
)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

# ====== EVAL ======
pred_logit = model.predict(valid_pool)
rmse = np.sqrt(mean_squared_error(valid_df[TARGET_COL].values, pred_logit))
print(f"\nRMSE(logit) valid: {rmse:.6f}")
p_hat = 1.0 / (1.0 + np.exp(-pred_logit))
if LABEL_COL in valid_df.columns and valid_df[LABEL_COL].notna().any():
    y_true = valid_df[LABEL_COL].astype(int).values
    pr_auc = average_precision_score(y_true, p_hat)
    print(f"PR-AUC valid (vs label): {pr_auc:.6f}")
    precisions, recalls, thresholds = precision_recall_curve(y_true, p_hat)
    f1s = (2 * precisions * recalls) / (precisions + recalls + 1e-12)
    best_idx = int(np.nanargmax(f1s))
    best_thr = float(thresholds[max(best_idx - 1, 0)]) if len(thresholds) else 0.5
    y_pred = (p_hat >= best_thr).astype(int)
    f1 = f1_score(y_true, y_pred)
    print(f"Best threshold (F1): {best_thr:.6f} | F1 valid: {f1:.6f}")
else:
    pr_auc = None
    best_thr = None
    f1 = None
    print("Label non presente (o vuota) -> salto metriche classificazione.")

# ====== SAVE ======
model.save_model(MODEL_OUT)
meta = {
    "input_file": IN_FILE,
    "text_col": TEXT_COL,
    "target_col": TARGET_COL,
    "label_col": LABEL_COL if LABEL_COL in df_feats.columns else None,
    "num_cols": num_cols,
    "feature_cols": feature_cols,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "use_borderline_weights": USE_BORDERLINE_WEIGHTS,
    "w_max": W_MAX if USE_BORDERLINE_WEIGHTS else None,
    "rmse_valid_logit": float(rmse),
    "pr_auc_valid": None if pr_auc is None else float(pr_auc),
    "best_threshold_valid": None if best_thr is None else float(best_thr),
    "f1_valid": None if f1 is None else float(f1),
    "model_file": MODEL_OUT
}
with open(META_OUT, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("\nSalvato modello:", MODEL_OUT)
print("Salvati metadata:", META_OUT)
print("Num features:", len(feature_cols), "| Numeric:", len(num_cols))
print("First feature names:", model.feature_names_[:10])
